# Project 4 — Milestone 1: Quantization Workload Augmentation

This notebook follows the AccelForge workflow from the lab:
1. define workload YAML (`iteration_space_shape`, `bits_per_value`, `einsums`)
2. load with `af.Spec.from_yaml(...)`
3. run `spec.evaluate_mapping()`
4. compare baseline vs quantized workload

## Milestone 1 tasks (from project_4.pdf)

- Study hardware cost of supporting quantization
- Create a compound component to model quantization area/energy
- Pick a workload and augment the spec with einsums that represent quantization

This notebook focuses on the third bullet using a minimal weight-only NVFP4-style model.

In [1]:
from pathlib import Path
import os
import yaml

try:
    import accelforge as af
    AF_AVAILABLE = True
except Exception as e:
    af = None
    AF_AVAILABLE = False
    print("accelforge import failed:", e)
    print("Notebook will still generate workload/mapping YAML files.")
    print("Run evaluation cells in your AccelForge-enabled environment.")


def find_lab_root() -> Path:
    cwd = Path.cwd()

    # Optional explicit override
    env_root = os.environ.get("AF_LAB4_ROOT")
    if env_root:
        return Path(env_root).expanduser()

    # If launched from lab_4 directly
    if (cwd / "project4_m1_quantization_workload.ipynb").exists() or (cwd / "project4_m1").exists():
        return cwd

    # Common nested layout: <repo>/workspace/lab_4
    nested = cwd / "workspace" / "lab_4"
    if nested.exists():
        return nested

    # Known local absolute path fallbacks
    known_paths = [
        Path("/Users/yichongzhang/Desktop/\u5927\u5b66/\u5927\u4e09\u4e0b/hardware design for AI accelerators/final project/Hardware-for-quantization/workspace/lab_4"),
        Path("/Users/bearxiong/Documents/MIT/Deep_Learning/Final-Project/workspace/lab_4"),
    ]
    for known in known_paths:
        if known.exists():
            return known

    return cwd


ROOT = find_lab_root()
OUT_DIR = ROOT / "project4_m1"
OUT_DIR.mkdir(exist_ok=True)

# Standalone architecture file (no part6 dependency)
ARCH_FILE = OUT_DIR / "arch_template.yaml"
# ========== Architecture Parameters (adjust these!) ==========
ARCH_PARAMS = dict(
    GLB_SIZE = 524288,    # GLB capacity in bits (524288 = 64KB)
    RF_SIZE  = 1024,      # RF capacity per PE in bits (1024 = 128B)
    PE_X     = 8,         # spatial X fanout (PE array width)
    PE_Y     = 8,         # spatial Y fanout (PE array height)
    DRAM_BW  = 8,         # DRAM actions/cycle
    GLB_BW   = 32,        # GLB actions/cycle
)
# Total PEs = PE_X * PE_Y
print(f"Architecture: GLB={ARCH_PARAMS['GLB_SIZE']//8//1024}KB, "
      f"RF={ARCH_PARAMS['RF_SIZE']//8}B/PE, "
      f"PEs={ARCH_PARAMS['PE_X']}x{ARCH_PARAMS['PE_Y']}="
      f"{ARCH_PARAMS['PE_X']*ARCH_PARAMS['PE_Y']}")


def dump_yaml(path: Path, obj: dict):
    with open(path, "w") as f:
        yaml.safe_dump(obj, f, sort_keys=False)


def evaluate(arch_file: Path, workload_file: Path, mapping_file: Path, jinja_data=None):
    if not AF_AVAILABLE:
        raise RuntimeError("accelforge is unavailable in this kernel.")
    for fp in [arch_file, workload_file, mapping_file]:
        if not Path(fp).exists():
            raise FileNotFoundError(f"Missing required file: {fp}")
    spec = af.Spec.from_yaml(str(arch_file), str(workload_file), str(mapping_file), jinja_parse_data=jinja_data)
    result = spec.evaluate_mapping()
    return result


print("Working directory:", Path.cwd())
print("Lab root:", ROOT)
print("Output directory:", OUT_DIR)
print("Architecture:", ARCH_FILE)

Architecture: GLB=64KB, RF=128B/PE, PEs=8x8=64
Working directory: /home/workspace/workspace/lab_4
Lab root: /home/workspace/workspace/lab_4
Output directory: /home/workspace/workspace/lab_4/project4_m1
Architecture: /home/workspace/workspace/lab_4/project4_m1/arch_template.yaml


## Step 1: Baseline dense GEMM workload

Start from a simple lab-style workload with one einsum.

In [2]:
baseline_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "k": "0 <= k < 128",
        },
        "bits_per_value": {
            "A": 16,
            "W": 16,
            "Y": 16,
        },
        "einsums": [
            {
                "name": "MatMul",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "k"], "density": 1.0},
                    {"name": "W", "projection": ["n", "k"], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            }
        ],
    }
}

baseline_workload_file = OUT_DIR / "workload_baseline.yaml"
baseline_mapping_file = OUT_DIR / "mapping_baseline.yaml"

baseline_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, W, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 8
  - !Temporal
    rank_variable: k
    tile_shape: 8
  - !Storage
    tensors: [A, W, Y]
    component: GLB
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: k
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: RF
  - !Compute
    einsum: MatMul
    component: FP4MAC
""".strip()

dump_yaml(baseline_workload_file, baseline_workload)
baseline_mapping_file.write_text(baseline_mapping_text)

print("Wrote baseline workload:", baseline_workload_file)
print("Wrote baseline mapping:", baseline_mapping_file)

Wrote baseline workload: /home/workspace/workspace/lab_4/project4_m1/workload_baseline.yaml
Wrote baseline mapping: /home/workspace/workspace/lab_4/project4_m1/mapping_baseline.yaml


In [3]:
if AF_AVAILABLE and ARCH_FILE.exists() and baseline_mapping_file.exists():
    baseline_result = evaluate(ARCH_FILE, baseline_workload_file, baseline_mapping_file, ARCH_PARAMS)
    print("Baseline mapping evaluation completed.")
    print(baseline_result)
else:
    print("Skipped baseline evaluate_mapping().")
    print("AF_AVAILABLE:", AF_AVAILABLE)
    print("arch exists:", ARCH_FILE.exists(), ARCH_FILE)
    print("mapping exists:", baseline_mapping_file.exists(), baseline_mapping_file)

Baseline mapping evaluation completed.


## Step 2–6: NVFP4-style weight-only augmentation

We now augment the workload spec with explicit quantization tensors and einsums:
- split `k -> kb, ki` with `ki=16`
- add quantized weights `Wq[n,kb,ki]`
- add block scales `Sw[n,kb]`
- add dequantized weights `Wdq[n,kb,ki]`
- add `DequantW` einsum before the main GEMM
- update GEMM to consume `Wdq`

In [4]:
quant_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "kb": "0 <= kb < 8",
            "ki": "0 <= ki < 16",
        },
        "bits_per_value": {
            "A": 16,
            "Wq": 4,
            "Sw": 16,
            "Wdq": 16,
            "Y": 16,
        },
        "einsums": [
            {
                "name": "DequantW",
                "tensor_accesses": [
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Wdq", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            {
                "name": "MatMulQ",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Wdq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            },
        ],
    }
}

quant_workload_file = OUT_DIR / "workload_nvfp4_weight_only.yaml"
dump_yaml(quant_workload_file, quant_workload)
print("Wrote quantized workload:", quant_workload_file)
with open(quant_workload_file) as f:
    print(f.read())

Wrote quantized workload: /home/workspace/workspace/lab_4/project4_m1/workload_nvfp4_weight_only.yaml
workload:
  iteration_space_shape:
    m: 0 <= m < 64
    n: 0 <= n < 64
    kb: 0 <= kb < 8
    ki: 0 <= ki < 16
  bits_per_value:
    A: 16
    Wq: 4
    Sw: 16
    Wdq: 16
    Y: 16
  einsums:
  - name: DequantW
    tensor_accesses:
    - name: Wq
      projection:
      - n
      - kb
      - ki
      density: 1.0
    - name: Sw
      projection:
      - n
      - kb
      density: 1.0
    - name: Wdq
      projection:
      - n
      - kb
      - ki
      output: true
  - name: MatMulQ
    tensor_accesses:
    - name: A
      projection:
      - m
      - kb
      - ki
      density: 1.0
    - name: Wdq
      projection:
      - n
      - kb
      - ki
      density: 1.0
    - name: Y
      projection:
      - m
      - n
      output: true



## Step 8: Validate with AccelForge (`from_yaml` + `evaluate_mapping()`)

We generate a first-pass quantized mapping that includes both einsums:
- `DequantW`
- `MatMulQ`

If mapping evaluation fails due scheduler constraints, the parser check still confirms workload structure validity and you can iterate the mapping next.

In [5]:
quant_mapping_file = OUT_DIR / "mapping_nvfp4_weight_only.yaml"
quant_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, Wq, Sw, Wdq, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 4
  - !Temporal
    rank_variable: kb
    tile_shape: 2
  - !Temporal
    rank_variable: ki
    tile_shape: 8
  - !Storage
    tensors: [A, Wq, Sw, Wdq, Y]
    component: GLB
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: kb
    tile_shape: 1
  - !Temporal
    rank_variable: ki
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: RF
  - !Compute
    einsum: DequantW
    component: QuantMAC
  - !Compute
    einsum: MatMulQ
    component: FP4MAC
""".strip()

quant_mapping_file.write_text(quant_mapping_text)
print("Wrote quantized mapping:", quant_mapping_file)

if AF_AVAILABLE and ARCH_FILE.exists() and quant_workload_file.exists() and quant_mapping_file.exists():
    # Parser validation + mapping evaluation
    spec_q = af.Spec.from_yaml(str(ARCH_FILE), str(quant_workload_file), str(quant_mapping_file), jinja_parse_data=ARCH_PARAMS)
    print("Quantized spec parsed successfully.")

    try:
        quant_result = spec_q.evaluate_mapping()
        print("Quantized mapping evaluation completed.")
        print(quant_result)
    except Exception as e:
        print("Quantized mapping parse succeeded, but evaluate_mapping() needs mapping refinement:")
        print(type(e).__name__, e)
else:
    print("Skipped Spec.from_yaml/evaluate_mapping().")
    print("AF_AVAILABLE:", AF_AVAILABLE)
    print("arch exists:", ARCH_FILE.exists(), ARCH_FILE)
    print("workload exists:", quant_workload_file.exists(), quant_workload_file)
    print("mapping exists:", quant_mapping_file.exists(), quant_mapping_file)

Wrote quantized mapping: /home/workspace/workspace/lab_4/project4_m1/mapping_nvfp4_weight_only.yaml
Quantized spec parsed successfully.
Quantized mapping parse succeeded, but evaluate_mapping() needs mapping refinement:
ValueError Component QuantMAC not found in flattened arch


## Full NVFP4 Quantization (W4A4) — Two-Level Scaling

NVIDIA's NVFP4 on Blackwell uses **two-level scaling**:
- **Per-tensor scale** (coarse): One FP32 scalar per entire tensor. `projection: []`.
- **Per-block scale** (fine): One FP8 (E4M3) scale per block of 16 elements along K.

### Quantization (FP16 → FP4):
1. `TensorScaleA`: $S_{ga} = \text{reduce}_{m,k_b,k_i}(|A|)$ — per-tensor scale (scalar)
2. `TensorQuantA`: $A_{scl} = A \times S_{ga}^{-1}$
3. `BlockScaleA`: $S_{ba}[m,k_b] = \text{reduce}_{k_i}(|A_{scl}|)$ — per-block scale
4. `BlockQuantA`: $A_q = A_{scl} \times S_{ba}^{-1}$ — quantize to FP4
5–8. Same for weights

### FP4 × FP4 Compute (FP32 accumulate):
9. `MatMulNVFP4`: $Y_{raw}[m,n,k_b] \mathrel{+}= A_q \times W_q$ — reduce $k_i$

### Rescale (2 inputs per einsum):
10. `RescaleBlockA`: $Y_{tmp} = Y_{raw} \times S_{ba}[m,k_b]$
11. `RescaleBlockW`: $Y_{blk} = Y_{tmp} \times S_{bw}[n,k_b]$
12. `RescaleTensorA`: $Y_{tmp2}[m,n] = \sum_{k_b} Y_{blk} \times S_{ga}$ — reduce $k_b$
13. `RescaleTensorW`: $Y = Y_{tmp2} \times S_{gw}$


In [6]:
nvfp4_full_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "kb": "0 <= kb < 8",
            "ki": "0 <= ki < 16",
        },
        "bits_per_value": {
            # Original full-precision inputs
            "A": 16,         # FP16 activations
            "W": 16,         # FP16 weights
            # Per-TENSOR scales — FP32, approximated as per-row/per-col
            # (scalar projection [] crashes AccelForge auto-mapper,
            #  so we use [m] / [n] instead — negligible overhead)
            "Sga": 32,       # Global scale for activations [m]
            "Sgw": 32,       # Global scale for weights [n]
            # Tensor-scaled intermediates
            "Ascl": 16,      # Activations after global scaling
            "Wscl": 16,      # Weights after global scaling
            # Per-BLOCK scales — FP8 E4M3
            "Sba": 8,        # Block scale for activations [m, kb]
            "Sbw": 8,        # Block scale for weights [n, kb]
            # Quantized — FP4 E2M1
            "Aq": 4,         # Quantized activations
            "Wq": 4,         # Quantized weights
            # FP32 accumulator
            "Yraw": 32,      # FP4*FP4 partial sums
            "Ytmp": 32,      # After activation block rescale
            "Yblk": 32,      # After weight block rescale
            "Ytmp2": 32,     # After activation tensor rescale
            # Final output
            "Y": 16,         # FP16 after all rescaling
        },
        "einsums": [
            # ===== Activation quantization =====
            # 1. Per-tensor scale: reduce kb, ki -> per-row scale
            {
                "name": "TensorScaleA",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sga", "projection": ["m"], "output": True},
                ],
            },
            # 2. Apply per-tensor scale
            {
                "name": "TensorQuantA",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sga", "projection": ["m"], "density": 1.0},
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "output": True},
                ],
            },
            # 3. Per-block scale: reduce ki
            {
                "name": "BlockScaleA",
                "tensor_accesses": [
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "output": True},
                ],
            },
            # 4. Quantize to FP4
            {
                "name": "BlockQuantA",
                "tensor_accesses": [
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "density": 1.0},
                    {"name": "Aq", "projection": ["m", "kb", "ki"], "output": True},
                ],
            },
            # ===== Weight quantization =====
            # 5. Per-tensor scale: reduce kb, ki -> per-col scale
            {
                "name": "TensorScaleW",
                "tensor_accesses": [
                    {"name": "W", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sgw", "projection": ["n"], "output": True},
                ],
            },
            # 6. Apply per-tensor scale
            {
                "name": "TensorQuantW",
                "tensor_accesses": [
                    {"name": "W", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sgw", "projection": ["n"], "density": 1.0},
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            # 7. Per-block scale: reduce ki
            {
                "name": "BlockScaleW",
                "tensor_accesses": [
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "output": True},
                ],
            },
            # 8. Quantize to FP4
            {
                "name": "BlockQuantW",
                "tensor_accesses": [
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            # ===== FP4 x FP4 MatMul, FP32 accumulate =====
            # 9. Yraw[m,n,kb] += Aq[m,kb,ki] * Wq[n,kb,ki]  (reduce ki)
            {
                "name": "MatMulNVFP4",
                "tensor_accesses": [
                    {"name": "Aq", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Yraw", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # ===== Rescale output (2 inputs each) =====
            # 10. Undo activation block scale
            {
                "name": "RescaleBlockA",
                "tensor_accesses": [
                    {"name": "Yraw", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "density": 1.0},
                    {"name": "Ytmp", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # 11. Undo weight block scale
            {
                "name": "RescaleBlockW",
                "tensor_accesses": [
                    {"name": "Ytmp", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Yblk", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # 12. Undo activation tensor scale + reduce kb
            {
                "name": "RescaleTensorA",
                "tensor_accesses": [
                    {"name": "Yblk", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sga", "projection": ["m"], "density": 1.0},
                    {"name": "Ytmp2", "projection": ["m", "n"], "output": True},
                ],
            },
            # 13. Undo weight tensor scale
            {
                "name": "RescaleTensorW",
                "tensor_accesses": [
                    {"name": "Ytmp2", "projection": ["m", "n"], "density": 1.0},
                    {"name": "Sgw", "projection": ["n"], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            },
        ],
    }
}

nvfp4_full_workload_file = OUT_DIR / "workload_nvfp4_full.yaml"
dump_yaml(nvfp4_full_workload_file, nvfp4_full_workload)
print("Wrote full NVFP4 workload:", nvfp4_full_workload_file)
with open(nvfp4_full_workload_file) as f:
    print(f.read())


Wrote full NVFP4 workload: /home/workspace/workspace/lab_4/project4_m1/workload_nvfp4_full.yaml
workload:
  iteration_space_shape:
    m: 0 <= m < 64
    n: 0 <= n < 64
    kb: 0 <= kb < 8
    ki: 0 <= ki < 16
  bits_per_value:
    A: 16
    W: 16
    Sga: 32
    Sgw: 32
    Ascl: 16
    Wscl: 16
    Sba: 8
    Sbw: 8
    Aq: 4
    Wq: 4
    Yraw: 32
    Ytmp: 32
    Yblk: 32
    Ytmp2: 32
    Y: 16
  einsums:
  - name: TensorScaleA
    tensor_accesses:
    - name: A
      projection:
      - m
      - kb
      - ki
      density: 1.0
    - name: Sga
      projection:
      - m
      output: true
  - name: TensorQuantA
    tensor_accesses:
    - name: A
      projection:
      - m
      - kb
      - ki
      density: 1.0
    - name: Sga
      projection:
      - m
      density: 1.0
    - name: Ascl
      projection:
      - m
      - kb
      - ki
      output: true
  - name: BlockScaleA
    tensor_accesses:
    - name: Ascl
      projection:
      - m
      - kb
      - ki
      dens

In [7]:
from accelforge.mapper import Metrics

nvfp4_full_mapping_file = OUT_DIR / "mapping_nvfp4_full.yaml"
nvfp4_full_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, W, Sga, Sgw, Ascl, Wscl, Sba, Sbw, Aq, Wq, Yraw, Ytmp, Yblk, Ytmp2, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 4
  - !Temporal
    rank_variable: kb
    tile_shape: 2
  - !Temporal
    rank_variable: ki
    tile_shape: 8
  - !Storage
    tensors: [A, W, Sga, Sgw, Ascl, Wscl, Sba, Sbw, Aq, Wq, Yraw, Ytmp, Yblk, Ytmp2, Y]
    component: GLB
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: kb
    tile_shape: 1
  - !Temporal
    rank_variable: ki
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: RF
  - !Compute
    einsum: TensorScaleA
    component: QuantMAC
  - !Compute
    einsum: TensorQuantA
    component: QuantMAC
  - !Compute
    einsum: BlockScaleA
    component: QuantMAC
  - !Compute
    einsum: BlockQuantA
    component: QuantMAC
  - !Compute
    einsum: TensorScaleW
    component: QuantMAC
  - !Compute
    einsum: TensorQuantW
    component: QuantMAC
  - !Compute
    einsum: BlockScaleW
    component: QuantMAC
  - !Compute
    einsum: BlockQuantW
    component: QuantMAC
  - !Compute
    einsum: MatMulNVFP4
    component: FP4MAC
  - !Compute
    einsum: RescaleBlockA
    component: RescaleMAC
  - !Compute
    einsum: RescaleBlockW
    component: RescaleMAC
  - !Compute
    einsum: RescaleTensorA
    component: RescaleMAC
  - !Compute
    einsum: RescaleTensorW
    component: RescaleMAC
""".strip()

nvfp4_full_mapping_file.write_text(nvfp4_full_mapping_text)
print("Wrote full NVFP4 mapping:", nvfp4_full_mapping_file)

if AF_AVAILABLE and ARCH_FILE.exists() and nvfp4_full_workload_file.exists():
    spec_nvfp4 = af.Spec.from_yaml(str(ARCH_FILE), str(nvfp4_full_workload_file), jinja_parse_data=ARCH_PARAMS)
    spec_nvfp4.mapper.metrics = Metrics.LATENCY | Metrics.ENERGY
    print("Full NVFP4 spec parsed successfully.")

    try:
        all_nvfp4_mappings = spec_nvfp4.map_workload_to_arch()
        print("Full NVFP4 auto-mapping completed.")
        print(f"Found {len(all_nvfp4_mappings.data)} mappings")

        # Select best mapping by minimum EDP (energy x latency)
        df = all_nvfp4_mappings.data
        energy_cols = [c for c in df.columns if 'energy' in c.lower()]
        latency_cols = [c for c in df.columns if 'latency' in c.lower()]
        print(f"Energy columns: {energy_cols}")
        print(f"Latency columns: {latency_cols}")

        best_idx = 0
        if energy_cols and latency_cols:
            edp = df[energy_cols[0]] * df[latency_cols[0]]
            best_idx = edp.idxmin()

        nvfp4_full_result = all_nvfp4_mappings[best_idx]
        print(f"Best mapping index: {best_idx}")
        print(f"Energy: {nvfp4_full_result.energy()} pJ")
        print(f"Latency: {nvfp4_full_result.latency()} cycles")

        # Export and save the auto-generated mapping
        best_mapping_yaml = nvfp4_full_result.mapping().to_yaml()
        auto_mapping_file = OUT_DIR / "mapping_nvfp4_full_auto.yaml"
        auto_mapping_file.write_text(best_mapping_yaml)
        print(f"\nSaved auto-generated mapping to: {auto_mapping_file}")
        print("\n=== Auto-generated mapping ===")
        print(best_mapping_yaml)
    except Exception as e:
        import traceback
        print("Full NVFP4 parse succeeded, but map_workload_to_arch() failed:")
        traceback.print_exc()
else:
    print("Skipped Spec.from_yaml/map_workload_to_arch().")
    print("AF_AVAILABLE:", AF_AVAILABLE)
    print("arch exists:", ARCH_FILE.exists(), ARCH_FILE)
    print("workload exists:", nvfp4_full_workload_file.exists(), nvfp4_full_workload_file)

Wrote full NVFP4 mapping: /home/workspace/workspace/lab_4/project4_m1/mapping_nvfp4_full.yaml
Full NVFP4 spec parsed successfully.


Getting energy, latency, and leak power for components running RescaleTensorW: 100%|██████████| 13/13 [00:00<00:00, 59.06it/s]
Generating pmapping templates for compute QuantMAC Einsum TensorScaleW: 40it [00:00, 197.13it/s]s]
Generating pmapping templates for compute QuantMAC Einsum TensorScaleA: 40it [00:00, 202.99it/s]
Generating pmapping templates for compute QuantMAC Einsum BlockScaleW: 65it [00:00, 158.14it/s]/s]
Generating pmapping templates for compute QuantMAC Einsum BlockScaleA: 65it [00:00, 157.69it/s]
Generating pmapping templates for compute FP4MAC Einsum TensorScaleW: 40it [00:00, 136.54it/s]/s]
Generating pmapping templates for compute FP4MAC Einsum TensorScaleA: 40it [00:00, 147.03it/s]]
Generating pmapping templates for compute RescaleMAC Einsum TensorScaleW: 40it [00:00, 224.77it/s]]
Generating pmapping templates for compute RescaleMAC Einsum TensorScaleA: 40it [00:00, 206.59it/s]
Generating pmapping templates for compute FP4MAC Einsum BlockScaleW: 65it [00:00, 216.59i

Einsum TensorScaleA has 120 pmapping templates:
	0	[A in DRAM] T-m  [Sga in GLB] T-kb  T-ki  T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  QuantMAC computes TensorScaleA
	1	[A in DRAM] T-m  [Sga in GLB] T-kb  T-ki  T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  [A in RF] QuantMAC computes TensorScaleA
	2	[A in DRAM] T-m  [Sga in GLB] T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  [Sga in RF] T-kb  T-ki  QuantMAC computes TensorScaleA
	3	[A in DRAM] T-m  [Sga in GLB] T-kb  T-ki  T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  [A in RF] [Sga in RF] T-kb  T-ki  QuantMAC computes TensorScaleA
	4	[A in DRAM] T-m  [Sga in GLB] T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  [Sga in RF] T-kb  T-ki  [A in RF] QuantMAC computes TensorScaleA
	5	[A in DRAM] T-m  [A in GLB] [Sga in GLB] T-kb  T-ki  T-m  S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-X-kb  QuantMAC computes TensorScaleA
	6	[A in DRAM] T-m  [Sga in GLB] T-kb  T-ki  T-m  [A in GLB] S-Y-m  S-Y-ki  S-Y-kb  S-X-m  S-X-ki  S-

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



	2244	[Ytmp2 in DRAM] [Sga in DRAM] T-kb  T-m  T-n  [Sga in GLB] T-kb  T-n  [Yblk in GLB] T-m  S-Y-n  S-Y-m  S-Y-kb  S-X-n  S-X-m  S-X-kb  [Sga in RF] T-kb  T-n  [Yblk in RF] FP4MAC computes RescaleTensorA
	2245	[Ytmp2 in DRAM] [Sga in DRAM] T-kb  T-m  T-n  [Sga in GLB] T-kb  T-n  [Yblk in GLB] T-kb  T-m  T-n  S-Y-n  S-Y-m  S-Y-kb  S-X-n  S-X-m  S-X-kb  [Yblk in RF] [Sga in RF] T-kb  T-n  FP4MAC computes RescaleTensorA
	2246	[Ytmp2 in DRAM] [Sga in DRAM] T-kb  T-m  T-n  [Sga in GLB] T-kb  T-n  [Yblk in GLB] T-m  S-Y-n  S-Y-m  S-Y-kb  S-X-n  S-X-m  S-X-kb  [Sga in RF] T-n  [Ytmp2 in RF] T-kb  FP4MAC computes RescaleTensorA
	2247	[Ytmp2 in DRAM] [Sga in DRAM] T-kb  T-m  T-n  [Sga in GLB] T-kb  T-n  [Yblk in GLB] T-m  T-n  S-Y-n  S-Y-m  S-Y-kb  S-X-n  S-X-m  S-X-kb  [Ytmp2 in RF] [Sga in RF] T-kb  T-n  FP4MAC computes RescaleTensorA
	2248	[Ytmp2 in DRAM] [Sga in DRAM] T-kb  T-m  T-n  [Sga in GLB] T-kb  T-n  [Yblk in GLB] T-kb  T-m  T-n  S-Y-n  S-Y-m  S-Y-kb  S-X-n  S-X-m  S-X-kb  [Yblk in

Generating pmappings:  42%|████▏     | 79/188 [07:44<05:18,  2.92s/it]/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Generating pmappings:  49%|████▉     | 93/188 [09:15<14:21,  9.07s/it]/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Grouping pmappings for RescaleTensorW: 100%|██████████| 6/6 [00:00<00:00, 176.71it/s]


MatMulNVFP4: 9.43e10 total, 626 (1/1.51e08) valid, 7.28e07 (1/1.30e03) evaluated, 1.44e04 (1/6.57e06) Pareto-Optimal
BlockQuantA: 1.12e09 total, 20 (1/5.60e07) valid, 1.77e07 (1/63) evaluated, 8.05e03 (1/1.39e05) Pareto-Optimal
BlockQuantW: 1.12e09 total, 20 (1/5.60e07) valid, 1.77e07 (1/63) evaluated, 8.05e03 (1/1.39e05) Pareto-Optimal
RescaleTensorA: 2.39e09 total, 18 (1/1.33e08) valid, 1.91e07 (1/125) evaluated, 4.64e03 (1/5.15e05) Pareto-Optimal
RescaleBlockA: 3.55e09 total, 11 (1/3.23e08) valid, 2.65e07 (1/134) evaluated, 8.65e03 (1/4.11e05) Pareto-Optimal
TensorQuantA: 2.43e08 total, 11 (1/2.21e07) valid, 6.20e06 (1/39) evaluated, 714 (1/3.40e05) Pareto-Optimal
RescaleBlockW: 3.55e09 total, 15 (1/2.37e08) valid, 2.73e07 (1/130) evaluated, 8.49e03 (1/4.18e05) Pareto-Optimal
TensorQuantW: 2.43e08 total, 11 (1/2.21e07) valid, 6.20e06 (1/39) evaluated, 714 (1/3.40e05) Pareto-Optimal
TensorScaleA: 2.42e06 total, 0 (1/inf) valid, 1.86e05 (1/13) evaluated, 9 (1/2.68e05) Pareto-Optimal
B

Compressing pmappings: 100%|██████████| 13/13 [00:12<00:00,  1.00it/s]
Joining pmappings for TensorScaleA <--> TensorQuantA (2/13): 100%|██████████| 31/31 [00:00<00:00, 73.15it/s] 
Joining pmappings for TensorQuantA <--> BlockScaleA (3/13): 100%|██████████| 129/129 [00:00<00:00, 599.80it/s]
Grouping pmappings: 100%|██████████| 9/9 [00:00<00:00, 278.37it/s]
Joining pmappings for BlockScaleA <--> BlockQuantA (4/13): 100%|██████████| 770/770 [00:08<00:00, 92.85it/s] 
Grouping pmappings: 100%|██████████| 133/133 [00:00<00:00, 296.18it/s]
Joining pmappings for BlockQuantA <--> TensorScaleW (5/13): 100%|██████████| 56/56 [00:00<00:00, 601.00it/s]
Joining pmappings for TensorScaleW <--> TensorQuantW (6/13): 100%|██████████| 440/440 [00:00<00:00, 685.75it/s]
Joining pmappings for TensorQuantW <--> BlockScaleW (7/13): 100%|██████████| 1768/1768 [00:10<00:00, 176.13it/s]
Grouping pmappings: 100%|██████████| 104/104 [00:00<00:00, 350.39it/s]
Joining pmappings for BlockScaleW <--> BlockQuantW (8/1

Full NVFP4 auto-mapping completed.
Found 5 mappings
Energy columns: ['Total<SEP>energy', 'Total<SEP>leak_energy', 'Total<SEP>dynamic_energy', 'TensorScaleA<SEP>energy<SEP>DRAM<SEP>A<SEP>read', 'TensorScaleA<SEP>energy<SEP>GLB<SEP>Sga<SEP>read', 'TensorScaleA<SEP>energy<SEP>GLB<SEP>Sga<SEP>write', 'TensorScaleA<SEP>energy<SEP>FP4MAC<SEP>None<SEP>compute', 'TensorScaleA<SEP>energy<SEP>DRAM<SEP>leak', 'TensorScaleA<SEP>energy<SEP>GLB<SEP>leak', 'TensorScaleA<SEP>energy<SEP>RF<SEP>leak', 'TensorScaleA<SEP>energy<SEP>QuantMAC<SEP>leak', 'TensorScaleA<SEP>energy<SEP>FP4MAC<SEP>leak', 'TensorScaleA<SEP>energy<SEP>RescaleMAC<SEP>leak', 'TensorQuantA<SEP>energy<SEP>DRAM<SEP>A<SEP>read', 'TensorQuantA<SEP>energy<SEP>GLB<SEP>Sga<SEP>read', 'TensorQuantA<SEP>energy<SEP>DRAM<SEP>Ascl<SEP>write', 'TensorQuantA<SEP>energy<SEP>FP4MAC<SEP>None<SEP>compute', 'TensorQuantA<SEP>energy<SEP>DRAM<SEP>leak', 'TensorQuantA<SEP>energy<SEP>GLB<SEP>leak', 'TensorQuantA<SEP>energy<SEP>RF<SEP>leak', 'TensorQuantA<S

## Next steps for milestone completion

1. Run all three workloads: **Baseline (FP16)**, **NVFP4 weight-only (W4A16)**, **NVFP4 full (W4A4)**
2. Run `evaluate_mapping()` for each and compare energy/latency
3. Add your quantization compound component and re-evaluate energy/latency
4. Compare the three configurations and report overhead tradeoffs

## Milestone Presentation Visualizations

These plots summarize baseline vs NVFP4-weight-only vs NVFP4-full (W4A4) results for presentation.

- Main chart: normalized energy and latency ($\text{quantized} / \text{baseline}$)
- Optional chart: absolute values (if extractable from mapping objects)
- A compact summary table is also printed

In [8]:
import matplotlib.pyplot as plt
import pandas as pd


def _maybe_get(obj, names):
    for n in names:
        if hasattr(obj, n):
            v = getattr(obj, n)
            if callable(v):
                try:
                    return v()
                except Exception:
                    pass
            else:
                return v
    return None


def extract_metrics(mapping_obj):
    """Best-effort extractor for AccelForge mapping result metrics.
    Returns dict with keys: energy_pj, latency_cycles.
    """
    out = {"energy_pj": None, "latency_cycles": None}
    if mapping_obj is None:
        return out

    # Direct/common fields
    out["energy_pj"] = _maybe_get(mapping_obj, [
        "energy_pj", "energy", "total_energy", "total_energy_pj"
    ])
    out["latency_cycles"] = _maybe_get(mapping_obj, [
        "latency_cycles", "latency", "cycles", "total_latency", "total_cycles"
    ])

    # Some objects expose dataframe-like containers
    for container_name in ["df", "dataframe", "mappings_df", "table"]:
        container = _maybe_get(mapping_obj, [container_name])
        if container is None:
            continue
        try:
            cols = [c.lower() for c in container.columns]
            if out["energy_pj"] is None:
                for c in ["energy_pj", "energy", "total_energy", "total_energy_pj"]:
                    if c in cols:
                        out["energy_pj"] = float(container.loc[:, container.columns[cols.index(c)]].min())
                        break
            if out["latency_cycles"] is None:
                for c in ["latency_cycles", "latency", "cycles", "total_latency", "total_cycles"]:
                    if c in cols:
                        out["latency_cycles"] = float(container.loc[:, container.columns[cols.index(c)]].min())
                        break
        except Exception:
            pass

    # Cast if possible
    for k in ["energy_pj", "latency_cycles"]:
        try:
            if out[k] is not None:
                out[k] = float(out[k])
        except Exception:
            out[k] = None

    return out


# Collect current run results (expects earlier cells to have run)
b = globals().get("baseline_result", None)
q = globals().get("quant_result", None)
nv = globals().get("nvfp4_full_result", None)

baseline_metrics = extract_metrics(b)
quant_metrics = extract_metrics(q)
nvfp4_full_metrics = extract_metrics(nv)

rows = [
    {
        "config": "baseline (FP16)",
        "energy_pj": baseline_metrics["energy_pj"],
        "latency_cycles": baseline_metrics["latency_cycles"],
    },
    {
        "config": "NVFP4 weight-only (W4A16)",
        "energy_pj": quant_metrics["energy_pj"],
        "latency_cycles": quant_metrics["latency_cycles"],
    },
    {
        "config": "NVFP4 full (W4A4)",
        "energy_pj": nvfp4_full_metrics["energy_pj"],
        "latency_cycles": nvfp4_full_metrics["latency_cycles"],
    },
]
summary_df = pd.DataFrame(rows)
display(summary_df)

# Normalized chart if all values are available
all_have_metrics = all(
    m["energy_pj"] is not None and m["latency_cycles"] is not None
    for m in [baseline_metrics, quant_metrics, nvfp4_full_metrics]
)

if all_have_metrics:
    configs = ["W4A16\n(weight-only)", "W4A4\n(NVFP4 full)"]
    norm_energy = [
        quant_metrics["energy_pj"] / baseline_metrics["energy_pj"],
        nvfp4_full_metrics["energy_pj"] / baseline_metrics["energy_pj"],
    ]
    norm_latency = [
        quant_metrics["latency_cycles"] / baseline_metrics["latency_cycles"],
        nvfp4_full_metrics["latency_cycles"] / baseline_metrics["latency_cycles"],
    ]

    import numpy as np
    x = np.arange(len(configs))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Normalized bar chart
    ax = axes[0]
    bars1 = ax.bar(x - width/2, norm_energy, width, label="Energy", color="#4C78A8")
    bars2 = ax.bar(x + width/2, norm_latency, width, label="Latency", color="#F58518")
    ax.axhline(1.0, linestyle="--", color="gray", linewidth=1, label="Baseline")
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.set_ylabel("Normalized (Quantized / Baseline)")
    ax.set_title("Normalized Energy & Latency vs FP16 Baseline")
    ax.legend()
    for bar_group in [bars1, bars2]:
        for bar in bar_group:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{bar.get_height():.3f}x", ha="center", va="bottom", fontsize=9)

    # Absolute energy comparison
    ax2 = axes[1]
    abs_labels = ["Baseline\n(FP16)", "W4A16", "W4A4"]
    abs_energy = [baseline_metrics["energy_pj"], quant_metrics["energy_pj"], nvfp4_full_metrics["energy_pj"]]
    colors = ["#72B7B2", "#E45756", "#9D755D"]
    bars = ax2.bar(abs_labels, abs_energy, color=colors)
    ax2.set_title("Absolute Energy (pJ)")
    ax2.set_ylabel("pJ")
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("Metrics are not fully extractable from current result objects.")
    print("Presentation fallback: use parse/evaluate success + printed YAMLs + screenshot of mapping success logs.")
    print("If needed, share the exact result object API and I can wire precise metric extraction.")

,config,energy_pj,latency_cycles
0,baseline (FP16),6320537.6,524288.0
1,NVFP4 weight-only (W4A16),NaN,NaN
2,NVFP4 full (W4A4),1766284.8,22656.0


Metrics are not fully extractable from current result objects.
Presentation fallback: use parse/evaluate success + printed YAMLs + screenshot of mapping success logs.
If needed, share the exact result object API and I can wire precise metric extraction.
